In [ ]:
# data analysis
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# data management
from ucimlrepo import fetch_ucirepo
from openmeteopy import OpenMeteo
from openmeteopy.hourly import HourlyHistorical
from openmeteopy.daily import DailyHistorical
from openmeteopy.options import HistoricalOptions
from openmeteopy.utils.constants import *
import pickle

# models
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr
import joblib

# HTTP
import requests

# system 
import os

# global parameters
name_data_folder = "data"
os.makedirs(name_data_folder, exist_ok=True)
name_uci_raw = "raw_EP"
name_meteo_raw = 'raw_METEO'
name_combined_raw = 'raw_combined'

# Part 1 - download data

We download data that contains
- energy usage of `UCL` 
- weather data from UCI Machine Learning Repository
for localization: `Sceaux Ile-de-France` ($48.77^\circ\mathrm{N}, 2.29^\circ\mathrm{E}$)

Methodology
- If data does not exits, download `UCL data` using `ucimlrepo` library, saving it using `pickle` library
- If data exits, load data
- Date, time format: we use `iso8601` (YYYY-MM-DDTHH:mm:ss±hh:mm)

About data:
- raw_EP
    1. **date: Date in format dd/mm/yyyy**
    2. **time: time in format hh:mm:ss**
    3. **global_active_power: household global minute-averaged active power (in kilowatt)**
        > Comment: This power is *consumed* by circuit - it does perform work - **Explained variable**
    4. global_reactive_power: household global minute-averaged reactive power (in kilowatt)
        > Comment: This power is *stored* in circuit - it does not perform work but it is necessary for infrastructure (transformers etc.)
    5. voltage: minute-averaged voltage (in volt)
    6. global_intensity: household global minute-averaged current intensity (in ampere)
    7. sub_metering_1: energy sub-metering No. 1 (in watt-hour of active energy). It corresponds to  the kitchen, containing mainly a dishwasher, an oven and a microwave (hot plates are not el ectric but gas powered).
    8. sub_metering_2: energy sub-metering No. 2 (in watt-hour of active energy). It corresponds to  the laundry room, containing a washing-machine, a tumble-drier, a refrigerator and a li ght.
    9. sub_metering_3: energy sub-metering No. 3 (in watt-hour of active energy). It corresponds to an electric water-heater and an air-conditioner.
- raw_METEO 
    1. **temperature_2m: Air temperature measured at 2 meters above ground (°C)**  
        > Comment: Directly affects heating/cooling demand — highly relevant.
    2. **relativehumidity_2m: Relative humidity at 2 meters (%)**  
        > Comment: Influences comfort and cooling demand — relevant.
    3. **dewpoint_2m: Dew point temperature at 2 meters (°C)**  
        > Comment: Related to humidity and cooling needs — relevant.
    4. **apparent_temperature: Perceived temperature considering wind and humidity (°C)**  
        > Comment: Strong proxy for human comfort — relevant.
    5. pressure_msl: Atmospheric pressure at mean sea level (hPa)
        > Comment: Weather indicator, weak direct link to electricity use.
    6. surface_pressure: Atmospheric pressure at surface level (hPa)
        > Comment: Similar to above, weak relevance.
    7. **cloudcover: Total cloud cover (%)**  
        > Comment: Affects solar radiation and daylight — relevant.
    8. **cloudcover_low: Low-level cloud cover (%)**  
        > Comment: Clouds on different levels differs in density - absorption - relevant.
    9. **cloudcover_mid: Mid-level cloud cover (%)**  
        > Comment: Same as above.
    10. **cloudcover_high: High-level cloud cover (%)**  
        > Comment: Same as above.
    11. **windspeed_10m: Wind speed at 10 meters (m/s)**  
        > Comment: Can affect cooling/heating indirectly — moderate relevance.
    12. windspeed_100m: Wind speed at 100 meters (m/s)
        > Comment: More relevant for wind energy production, not household demand.
    13. winddirection_10m: Wind direction at 10 meters (°)
        > Comment: Very weak link to electricity consumption.
    14. winddirection_100m: Wind direction at 100 meters (°)
        > Comment: Same as above.
    15. **windgusts_10m: Maximum wind gusts at 10 meters (m/s)**  
        > Comment: Weather temperature translation indicator, moderate link to demand.
    16. **shortwave_radiation: Incoming shortwave solar radiation (W/m²)**  
        > Comment: Strongly affects daylight and solar heating — relevant.
    17. **direct_radiation: Direct solar radiation (W/m²)**  
        > Comment: Relevant for solar exposure and PV production.
    18. **diffuse_radiation: Diffuse solar radiation (W/m²)**  
        > Comment: Same as above.
    19. **vapor_pressure_deficit: Difference between saturation and actual vapor pressure (kPa)**  
        > Comment: Technical humidity measure, indirect relevance.
    20. et0_fao_evapotranspiration: Reference evapotranspiration (mm/day) 
        > Comment: Agricultural metric, weak relevance for household electricity.
    21. **precipitation: Total precipitation (mm)**  
        > Comment: Rainfall affects daylight and comfort — moderate relevance.
    22. **weathercode: Encoded weather condition (categorical)**  
        > Comment: Useful categorical proxy for overall weather — relevant.
    23. **snowfall: Snowfall amount (mm)**  
        > Comment: Strongly affects heating demand — relevant.
    24. soil_moisture_0_to_7cm: Soil moisture in top layer (m³/m³)
        > Comment: Agricultural metric, no direct link to household demand.
    25. soil_moisture_7_to_28cm: Soil moisture (m³/m³) 
        > Comment: Same as above.
    26. soil_moisture_28_to_100cm: Soil moisture (m³/m³)  
        > Comment: Same as above.
    27. soil_moisture_100_to_255cm: Soil moisture (m³/m³)  
        > Comment: Same as above.
    28. soil_temperature_0_to_7cm: Soil temperature (°C)  
        > Comment: Agricultural relevance, weak link to household demand.
    29. soil_temperature_7_to_28cm: Soil temperature (°C)  
        > Comment: Same as above.
    30. soil_temperature_28_to_100cm: Soil temperature (°C)  
        > Comment: Same as above.
    31. soil_temperature_100_to_255cm: Soil temperature (°C)  
        > Comment: Same as above.

Where, **bold** variables are relevant for us.

In [ ]:
### ----------- UCI ----------- ###

# flag 
exists = any(filename.startswith(name_uci_raw) for filename in os.listdir(name_data_folder))

# load data
if exists:
    print(f'{name_uci_raw} already exists')
    print(f"{name_uci_raw} Data loading...")
    ## opening file
    with open(f"{name_data_folder}/{name_uci_raw}.pkl", "rb") as file:
        ## loading file
        df_raw_EP = pickle.load(file)
    print(f"Done")
# download data
else:
    print(f"Downloading {name_uci_raw} data...")
    ## downloading data
    df_raw_EP = fetch_ucirepo(id=235)       
    print(f"Download complete \nSaving data...") 
    ## creating file for writing uci repo
    with open(f"{name_data_folder}/{name_uci_raw}.pkl", 'wb') as file:
        ## saving file
        pickle.dump(df_raw_EP, file)
    print('Done')

# leak check 
num_instances_metadata = df_raw_EP.metadata['num_instances']
num_intances_local = df_raw_EP.data.features.shape[0]
if num_intances_local != num_instances_metadata:
    print(f'''There is leak of data in your data set:
          - global data instances = {num_instances_metadata}
          - local data instances = {num_intances_local}
        Removing local dataset ...
          ''')
    os.remove(f"{name_data_folder}/{name_uci_raw}")
    print('Local dataset removed\nRun cell again')
else:
    print(f"\n---There is no leak in data---\n")

# load UCI data into df
df_EP = df_raw_EP.data.features
# variable information 
print(df_raw_EP.variables)

In [ ]:
### ----------- JOIN DATASETS I ----------- ###

# create final df
## UCI DATA
df_EP_copy = df_EP.copy()
### update `Date` and `Time` column to `datetime` type
df_EP_copy['Date'] = df_EP_copy['Date'] + ' ' + df_EP_copy['Time']
df_EP_copy['Date'] = pd.to_datetime(df_EP_copy['Date'], format='%d/%m/%Y %H:%M:%S', utc=True)
df_EP_copy = df_EP_copy.drop('Time', axis=1)
### set time index
df_EP_copy = df_EP_copy.set_index('Date')
### set float64 data type for rest columns 
df_EP_copy  = df_EP_copy.apply(pd.to_numeric, errors="coerce")
### We need to change Date in out `df` file - we need to get rid off minutes
df_EP_copy = df_EP_copy.resample('H').mean()
df_EP_copy

In [ ]:
### ----------- METEO ----------- ###

# parameters - Sceux Ile de France
longitude = 48.7764 # 48.66 N
latitude =  2.2903 # 2.29 E 

exists = any(filename.startswith(name_meteo_raw) for filename in os.listdir(name_data_folder))

# load data
if exists:
    print(f'{name_meteo_raw} already exists')
    print(f"{name_meteo_raw} Data loading...")
    ## open file
    with open(f"{name_data_folder}/{name_meteo_raw}.pkl", "rb") as file:
        ## load data
        df_raw_meteo = pickle.load(file)
    print(f"Done")
# download data 
else:
    try:
        start_date, end_date = str(df_EP_copy.index.date.min()), str(df_EP_copy.index.date.max())
    except:
        raise 'To download name_meteo_raw, you need run earlier cells'
    print(f"Downloading {name_meteo_raw} data...")
    ## set download options
    hourly = HourlyHistorical()
    daily = DailyHistorical()
    options = HistoricalOptions(latitude,
                                longitude, 
                                start_date=start_date,
                                end_date=end_date)
    ## download 
    ##TODO - check if we can skip minutes (redundant)
    df_raw_meteo = OpenMeteo(options, hourly.all(), daily.all())
    print(f"Download complete \nSaving data...") 
    ## saving file
    with open(f"{name_data_folder}/{name_meteo_raw}.pkl", "wb") as file:
        pickle.dump(df_raw_meteo, file)
    print('Done')
# Download data
df_meteo = df_raw_meteo.get_pandas()[0]
df_meteo

In [ ]:
### ----------- JOIN DATASETS II ----------- ###

## METEO DATA
df_meteo_copy = df_meteo.copy()
df_meteo_copy = df_meteo_copy.rename_axis('Date')
df_meteo_copy.index = pd.to_datetime(df_meteo_copy.index, format="%Y-%m-%dT%H:%M", utc=True)
## MERGE DATA
### joining 
df = df_EP_copy
df = df.join(df_meteo_copy, how='outer')
df
### saving data
df.to_json(f"{name_data_folder}/{name_combined_raw}.json")

# Part 2 - data engineering 

Methodology 
- Reduce meaningless data
    - We reduce features from `raw_METEO` that are meaningless
    - Selected features are **bolded** in [Part 1 - raw_METEO](#Part-1---download-data)
- Variable selection:
    - explained column: `Global_active_power` from `raw_EP`
    - explainable columns: Columns from `raw_METEO`
- Resampling data: *YYYY/MM/DD h* $\rightarrow$ *YYYY/MM/DD*
    - we aggregate by *mean*
- Remove missing rows
    - We remove all rows with NaN values
- Splitting data into two different `DataFrames` objects:
    - explained data $\rightarrow$ `y`
    - explainable data $\rightarrow$ `X`
- Feature engineering 
    - Box-Cox transformation
        - Use box-cox power transform in **y_train**
    - PCR reduction
        - Centralization and normalization: (Meeting theorem assumption)
        - PCR reduction with explainability(? - misspelled ?) threshold at $0.95\%$ 
    - PLS reduction
        - Centralization and normalization: (Meeting theorem assumption)
        - PCR reduction with 8 components (same as PCR - we should run it first)

In [ ]:
### ----------- data preparation ----------- ###

# loading data
df = pd.read_json(f"{name_data_folder}/{name_combined_raw}.json")

# parameters 
## target & features
explained_column = 'Global_active_power'
explainable_columns = [
    "temperature_2m",        # 1
    "relativehumidity_2m",   # 2
    "dewpoint_2m",           # 3
    "apparent_temperature",  # 4
    "cloudcover",            # 7
    "cloudcover_low",        # 8
    "cloudcover_mid",        # 9
    "cloudcover_high",       # 10
    "windspeed_10m",         # 11
    "windgusts_10m",         # 15
    "shortwave_radiation",   # 16
    "direct_radiation",      # 17
    "diffuse_radiation",     # 18
    "vapor_pressure_deficit",# 19
    "precipitation",         # 21
    "weathercode",           # 22
    "snowfall"               # 23
]
## Data selection params:
pc_explainability = 0.95
## train_test split params:
param_train_size = 0.8

# TODO - criteria selection
print('Columns selection...\n')
## select columns
mask_columns = [explained_column] + explainable_columns
## reduce df  
df = df[mask_columns]

# aggregate datetime data to day by mean
print('Resampling data...\n')
df = df.resample('d').mean()

# remove missing rows
mask_notna = df.notna().all(axis=1)
print(f'Removing missing rows...\n-from {df.shape[0]} rows {df.shape[0]-mask_notna.sum()} were removed\n')
df = df[mask_notna]

# split df to explained (`y`) and explanatory (`X`) variables
y = df[[explained_column]]
X = df.drop(columns=explained_column)
n, m = X.shape
print(f"X have shape {n}x{m}")

#### Splitting data

In [ ]:
# Split data
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X, y, 
    test_size=1-param_train_size, train_size=param_train_size, 
    random_state=42
)

# check sizes
if X_train_raw.shape[0] != y_train_raw.shape[0]:
    raise 'Dimensions do not match'

In [ ]:
### ----------- y - transformation - box-cos ----------- ###

pt = PowerTransformer(method='box-cox')  
y_train = pt.fit_transform(y_train_raw)

In [ ]:
### ----------- data selection - PCR ----------- ###

## PCR reduction
print("PCR...")
### standardization 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)
### PCR reduction 
pca = PCA(n_components=pc_explainability)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)
### Recreate Dataframe with same index 
cols = [f"PC{i+1} - {var:.4f}" for i, var in enumerate(pca.explained_variance_ratio_)]
X_train = pd.DataFrame(X_train_pca, index=X_train_raw.index, columns=cols)
X_test  = pd.DataFrame(X_test_pca,  index=X_test_raw.index,  columns=cols)
print(f"Results:\nFeatures: {X_train_raw.shape[1]} -> {X_train.shape[1]}\nExplainability:\n{pca.explained_variance_ratio_}")

X_train

In [ ]:
### ----------- data selection - partial least squares ----------- ###

## partial least square regression 
print("PLS...")
### standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)
### PLS reduction
pls = PLSRegression(n_components=8)
X_train_pls = pls.fit_transform(X_train_scaled, y_train)
X_test_pls = pls.transform(X_test_scaled)
### Recreate Dataframe with same index 
#### Calculate variance
expl_var_x = np.var(pls.x_scores_, axis=0) / np.var(X_train_scaled, axis=0).sum()
#### Name columns
cols = [f"PLS{i+1} - {var_x:.4f}" 
        for i, var_x in enumerate(expl_var_x)]
#### 
X_train = pd.DataFrame(X_train_pls[0], index=X_train_raw.index, columns=cols, dtype=float)
X_test  = pd.DataFrame(X_test_pls,  index=X_test_raw.index,  columns=cols, dtype=float)
print(f"Results:\nFeatures: {X_train_raw.shape[1]} -> {X_train.shape[1]}\nExplainability:\n{pls.x_scores_[-1]}")
X_train


# Part 3 - training regression models
In this section, models are trained to predict energy consumption based on selected weather data. The first model is a simple linear regression, and the second is a polynomial regression using ridge regularization. We perform 5-fold cross validation to select the best polynomial degree and ridge coefficient.

In [ ]:
# Simple linear regression
lr = LinearRegression()
lr.fit(X_train, y_train)


# Polynomial regression with ridge regularization grid search
poly_ridge = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('ridge', Ridge())
])

param_grid = {
    'poly__degree': [1,2,3,4],
    'ridge__alpha': np.logspace(-4, 4, 20)
}

kf = KFold(n_splits=5, shuffle=True, random_state=47)
grid = GridSearchCV(poly_ridge, param_grid, cv=kf, scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)
best_model = grid.best_estimator_

In [ ]:
# Results
## LR
y_pred_lr = pt.inverse_transform(lr.predict(X_test))
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)
pcc_lr, _ = pearsonr(y_test.to_numpy().reshape(-1), y_pred_lr.reshape(-1))

## POLY_RIDGE
y_pred_poly = pt.inverse_transform(best_model.predict(X_test).reshape(-1, 1))
mse_poly = mean_squared_error(y_test, y_pred_poly)
r2_poly = r2_score(y_test, y_pred_poly)
pcc_poly, _ = pearsonr(y_test.to_numpy().reshape(-1), y_pred_poly.reshape(-1))

## Results
best_params = grid.best_params_
print('Best parameters for polynomial ridge regression:')
print(f'Degree: {best_params["poly__degree"]}, Ridge alpha: {best_params["ridge__alpha"]:.2f}')
print('\nModels performance:')
print('Model\t\tMSE\t\tR2\t\tPCC')
print(f'Linear\t\t{mse_lr:.6f}\t{r2_lr:.6f}\t{pcc_lr:.6f}')
print(f'Polynomial\t{mse_poly:.6f}\t{r2_poly:.6f}\t{pcc_poly:.6f}')

In [ ]:
# Save models
joblib.dump(lr, "models/lr_model.pkl")
joblib.dump(best_model, "models/poly_model.pkl")
joblib.dump(pt, "models/pt_transform.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(pls, "models/pls.pkl")   # or PCA

In [ ]:
# Plot models predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].scatter(y_test, y_pred_lr, alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()],
            'r--')   # y=x line
axes[0].set_title(f"Linear regression (PCC = {pcc_lr:.3f})")

axes[1].scatter(y_test, y_pred_poly, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()],
            'r--')
axes[1].set_title(f"Polynomial regression (PCC = {pcc_poly:.3f})")

fig.supxlabel("Observed values")
fig.supylabel("Predicted values")
fig.suptitle("Predicted vs. observed values both models - PCR", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Plot models predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].scatter(y_test, y_pred_lr, alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()],
            'r--')   # y=x line
axes[0].set_title(f"Linear regression (PCC = {pcc_lr:.3f})")

axes[1].scatter(y_test, y_pred_poly, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()],
            'r--')
axes[1].set_title(f"Polynomial regression (PCC = {pcc_poly:.3f})")

fig.supxlabel("Observed values")
fig.supylabel("Predicted values")
fig.suptitle("Predicted vs. observed values both models - PLS", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Plot models predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].scatter(y_test, y_pred_lr, alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()],
            'r--')   # y=x line
axes[0].set_title(f"Linear regression (PCC = {pcc_lr:.3f})")

axes[1].scatter(y_test, y_pred_poly, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()],
            'r--')
axes[1].set_title(f"Polynomial regression (PCC = {pcc_poly:.3f})")

fig.supxlabel("Observed values")
fig.supylabel("Predicted values")
fig.suptitle("Predicted vs. observed values both models - BOX-COX", fontsize=14)
plt.tight_layout()
plt.show()

#### Compare target variable distribution between train and test set

In [ ]:
pd.DataFrame(y_train, columns=['Global_active_power']).describe()

In [ ]:
y_test.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].hist(y_train, bins=20, density=True)
axes[0].set_title(f"Training data distribution")

axes[1].hist(y_test, bins=20, density=True)
axes[1].set_title(f"Test data distribution")

fig.suptitle("Comparison of target variable distribution", fontsize=14)
plt.tight_layout()
plt.show()

Given the polynomial model results being better than the linear one, we can state that there are more complex dependencies in our data than simply linear ones. Nevertheless, even the polynomial model's performance is poor on this data. We can see that for both models, the predicted values are all focused around 1.0, which shows that the models struggle to identify the variation in the data and features that influence extreme (high or low) energy consumption values. This is not caused by the difference in the distribution of this variable on the training set, as can be seen in the histograms above, but rather by too little data or the use of an inappropriate model.

# Part 4 - Model sharing

For the code below to work, first run `fastapi dev app/main.py` in the terminal.

In [ ]:
def predict_energy(**params):
    url = "http://127.0.0.1:8000/predict-energy"
    response = requests.post(url, json=params)
    return response.json()

# Example usage
data = df.loc['2010-11-18', explainable_columns].to_dict()
predict_energy(**data)